In [70]:
import requests, html2text, mygene, pickle, json
from bs4 import BeautifulSoup

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

def get_gene_description_from_ncbi(gene_number):
    url = f"https://www.ncbi.nlm.nih.gov/gene/{gene_number}"
    try:
        response = requests.get(url, timeout=30, headers=headers)
        if response.status_code == 200:
            html = BeautifulSoup(response.content, 'html.parser') # Parse the HTML content of the page
            summary_tab = html.find('section', {'class': 'rprt-section gene-summary'}) # Find the "summary" tab content by inspecting the page's structure
            if summary_tab:
                html_to_text = html2text.HTML2Text() # Convert the HTML to plain text using html2text
                html_to_text.ignore_links = True  # Ignore hyperlinks in the summary text
                summary_text = html_to_text.handle(str(summary_tab)) # Extract the plain text from the "summary" tab
                parts_to_remove = ["##  Summary\n", "NEW", "Try the newGene table", "Try the newTranscript table", "**", "\nGo to the top of the page Help\n", "\n"]
                for part in parts_to_remove:
                    summary_text = summary_text.replace(part, ' ') # Remove the specified parts from the original text
                summary_text = ' '.join(summary_text.split()) # Reduce multiple spaces into one space
                return((summary_text, html)) # Print or save the extracted text
            else:
                print("Summary tab not found on the page.")
                return('', None)
        else:
            print(f"Failed to retrieve the webpage. Status code: {response.status_code}.")
            return('', None)
    except requests.exceptions.Timeout:
        print('Request timeout.')
        return('', None)
    
    
def get_protein_description_from_uniprot(protein_entry):
    url = f"https://rest.uniprot.org/uniprotkb/{protein_entry}.json"
    try:
        response = requests.get(url, timeout=30, headers=headers)
        if response.status_code == 200:
            data = response.json() 
            # json.dump(data, open(f"./input_data/uniprot_protein_entry_{protein_entry}.json", 'w'), indent=2)
            function_info = ''
            if 'comments' in data:
                for comment in data['comments']:
                    if comment.get('commentType') == 'FUNCTION':
                        for text in comment['texts']:
                            if 'value' in text:
                                function_info = function_info + text['value'] + ' '
            # print("No protein information sections found on the page.") if function_info == '' else None
            return (function_info, data)
        else:
            print(f"Failed to retrieve the webpage. Status code: {response.status_code}")
            return ('', None)
    except requests.exceptions.Timeout:
        print('Request timeout.')
        return ('', None)

In [96]:
mg = mygene.MyGeneInfo() # Reference: https://mygene.info/; https://pypi.org/project/mygene/; https://docs.mygene.info/projects/mygene-py/en/latest/
gene_query_results = mg.querymany(['CD24'], scopes='symbol', fields='all', species='human')
# json.dump(gene_query_results, open(f"./input_data/gene_query_results.json", 'w'), indent=2)
print(gene_query_results[0])
print('Gene ID:', gene_query_results[0]['_id'])
print('Gene Symbol:', gene_query_results[0]['symbol'])
print('Gene Full Name:', gene_query_results[0]['name'])
print('Gene Description:', gene_query_results[0]['summary'])
print('Gene Type:', gene_query_results[0]['type_of_gene'])
# Protein information related to the gene (Swiss-Prot)
if type(gene_query_results[0]['uniprot']['Swiss-Prot']) == list:
    protein_entry = gene_query_results[0]['uniprot']['Swiss-Prot'][0]
    print('Protein Entry:', gene_query_results[0]['uniprot']['Swiss-Prot'][0])
else:
    protein_entry = gene_query_results[0]['uniprot']['Swiss-Prot']
    print('Protein Entry:', gene_query_results[0]['uniprot']['Swiss-Prot'])
print('Protein Name:', get_protein_description_from_uniprot(protein_entry=protein_entry)[1]['proteinDescription']['recommendedName']['fullName']['value'])
print('Protein Function:', get_protein_description_from_uniprot(protein_entry=protein_entry)[0])


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


{'query': 'CD24', 'AllianceGenome': '1645', 'HGNC': '1645', 'MIM': '600074', '_id': '100133941', '_score': 19.219818, 'accession': {'genomic': ['ABBA01026189.1', 'CP068272.2', 'FJ226006.1', 'FP885534.9', 'JC231858.1', 'JN036721.1', 'JN565036.1', 'JN565037.1', 'JN565038.1', 'JN565039.1', 'JN565040.1', 'JN565041.1', 'JN565042.1', 'NC_000006.12', 'NC_060930.1', 'NG_041768.1', 'Y14692.1'], 'protein': ['AAA35665.1', 'AAB58807.1', 'AAH64619.1', 'AAP36068.1', 'ACI46150.1', 'AFJ05147.1', 'CAA49195.1', 'CDM22271.1', 'NP_001278666.1', 'NP_001278667.1', 'NP_001278668.1', 'NP_001346013.1', 'NP_037362.1', 'P25063.3', 'XP_054209946.1'], 'rna': ['AK026603.1', 'AK125531.1', 'AL535013.3', 'AU125939.1', 'BC064619.1', 'BG260536.1', 'BG755979.1', 'BT007404.1', 'BU580553.1', 'D87667.1', 'DA253943.1', 'DA378105.1', 'DA518957.1', 'DA521039.1', 'DA745135.1', 'DA753855.1', 'DQ530230.1', 'DQ530231.1', 'DQ530232.1', 'DQ530233.1', 'DQ530234.1', 'L33930.1', 'M58664.1', 'NM_001291737.1', 'NM_001291738.1', 'NM_00129

In [ ]:
# Load genes used in scGPT (https://drive.google.com/drive/folders/1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y)
with open(f"./input_data/vocab.json", 'rb') as handle:
    vocab_gene = json.load(handle)
gene_list = list(vocab_gene.keys())

# # Load genes used in Geneformer (https://huggingface.co/ctheodoris/Geneformer)
# with open(f"./input_data/token_dictionary.pkl", 'rb') as handle:
#     token_dictionary = pickle.load(handle)
# gene_list = list(token_dictionary.keys())

gene_list_results = mg.querymany(sorted(gene_list), scopes='symbol,ensembl.gene', fields='all', species='human')
gene_description_dict = {}
for result in gene_list_results:
    if "query" in result and "_id" in result and "symbol" in result and "name" in result and "summary" in result and "uniprot" in result and "Swiss-Prot" in result['uniprot']:
        gene_symbol = result['symbol']
        gene_full_name = result['name']
        gene_description = result['summary']
        gene_protein_entry = result['uniprot']['Swiss-Prot']
        gene_protein_name = get_protein_description_from_uniprot(protein_entry=gene_protein_entry)[1]['proteinDescription']['recommendedName']['fullName']['value']
        gene_protein_function = get_protein_description_from_uniprot(protein_entry=gene_protein_entry)[0]
        gene_description_dict[gene_symbol] = gene_full_name + ': ' + gene_description + '; ' + gene_protein_name + ': ' + gene_protein_function
        print(gene_symbol, ': ', gene_description_dict[gene_symbol])

json.dump(gene_description_dict, open(f"./input_data/gene_name_to_summary_page_scgpt.json", 'w'), indent=0)
# json.dump(gene_description_dict, open(f"./input_data/gene_name_to_summary_page_geneformer.json", 'w'), indent=0)

# Look at: ./input_data/GenePT_embedding/NCBI_cleaned_summary_of_genes.json
# Look at: ./input_data/GenePT_embedding_v2/NCBI_summary_of_genes.json
# Look at: ./input_data/GenePT_embedding_v2/NCBI_UniProt_summary_of_genes.json